# Lesson 13 — Ridge Regression
**Project:** T20 Final Score Predictor  
**Dataset:** `../data/t20_scores.csv`  
**Objective:** Linear Regression aur Ridge Regression ko same train/test data par compare karna.

Is notebook ko existing project ke `notebooks/` folder mein rakho.

### Cell 1 — Imports

In [1]:
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score

### Cell 2 — Dataset + X/y + Train/Test Split

In [2]:
df = pd.read_csv("../data/t20_scores.csv")

X = df[["venue", "runs_10", "wickets_10"]]
y = df["final_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Dataset:", df.shape)
print("Train:", X_train.shape)
print("Test:", X_test.shape)
df.head()

Dataset: (300, 4)
Train: (240, 3)
Test: (60, 3)


,venue,runs_10,wickets_10,final_score
0,Karachi,75,3,144
1,Multan,101,4,160
2,Rawalpindi,55,4,85
3,Lahore,105,0,220
4,Lahore,85,4,135


### Cell 3 — Preprocessing
`runs_10`, `wickets_10` scale honge aur `venue` one-hot encode hoga. Preprocessing ki fitting har Pipeline ke andar sirf training data par hogi.

In [3]:
num_cols = ["runs_10", "wickets_10"]
cat_cols = ["venue"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

### Cell 4 — Linear Regression aur Ridge Pipelines
`alpha=1.0` Ridge ki regularization strength hai. `clone()` se dono models ko independent preprocessors milte hain.

In [4]:
linear_pipe = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", LinearRegression()),
])

ridge_pipe = Pipeline([
    ("preprocess", clone(preprocessor)),
    ("model", Ridge(alpha=1.0)),
])

### Cell 5 — Linear Regression Train + Evaluate

In [5]:
linear_pipe.fit(X_train, y_train)
linear_pred = linear_pipe.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_r2 = r2_score(y_test, linear_pred)

print(f"Linear MAE: {linear_mae:.2f}")
print(f"Linear R²:  {linear_r2:.3f}")

Linear MAE: 9.99
Linear R²:  0.845


### Cell 6 — Ridge Train + Evaluate

In [6]:
ridge_pipe.fit(X_train, y_train)
ridge_pred = ridge_pipe.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_r2 = r2_score(y_test, ridge_pred)

print(f"Ridge MAE: {ridge_mae:.2f}")
print(f"Ridge R²:  {ridge_r2:.3f}")

Ridge MAE: 9.98
Ridge R²:  0.846


### Cell 7 — Models ka Comparison
**MAE kam** aur **R² zyada** generally better. Ridge har dataset par Linear Regression se better hona guaranteed nahi.

In [7]:
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge (alpha=1.0)"],
    "MAE": [linear_mae, ridge_mae],
    "R2": [linear_r2, ridge_r2],
})
comparison.round(3)

,Model,MAE,R2
0,Linear Regression,9.994,0.845
1,Ridge (alpha=1.0),9.982,0.846


### Cell 8 — New Match ka Prediction
10 overs ke baad runs, wickets aur venue dekar final total ka estimate.

In [8]:
new_match = pd.DataFrame({
    "venue": ["Karachi"],
    "runs_10": [85],
    "wickets_10": [2],
})

print("Linear predicted score:", round(linear_pipe.predict(new_match)[0]))
print("Ridge predicted score:", round(ridge_pipe.predict(new_match)[0]))

Linear predicted score: 161
Ridge predicted score: 161


### Cell 9 — Coefficients Compare
Ridge ki L2 regularization weights ko shrink karti hai. One-hot columns ki wajah se individual coefficients ko context mein interpret karna chahiye.

In [9]:
feature_names = ridge_pipe.named_steps["preprocess"].get_feature_names_out()

coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Linear": linear_pipe.named_steps["model"].coef_,
    "Ridge": ridge_pipe.named_steps["model"].coef_,
})
coefficients.round(3)

,Feature,Linear,Ridge
0,num__runs_10,29.033,28.905
1,num__wickets_10,-15.629,-15.564
2,cat__venue_Karachi,0.064,0.069
3,cat__venue_Lahore,-5.071,-4.989
4,cat__venue_Multan,-8.755,-8.591
5,cat__venue_Rawalpindi,13.763,13.512


### Cell 10 — Alpha ki Strength Samjho (Training Data Only)
Ye **sirf regularization effect samajhne** ke liye hai. Test-set metrics dekh kar best alpha select **mat** karo; baad mein cross-validation se tune karenge.

In [10]:
alpha_results = []

for alpha in [0.1, 1.0, 10.0, 100.0]:
    model = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", Ridge(alpha=alpha)),
    ])
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    weights = model.named_steps["model"].coef_

    alpha_results.append({
        "Alpha": alpha,
        "Train_MAE": mean_absolute_error(y_train, train_pred),
        "Sum_Abs_Weights": abs(weights).sum(),
    })

pd.DataFrame(alpha_results).round(3)

,Alpha,Train_MAE,Sum_Abs_Weights
0,0.1,9.444,72.245
1,1.0,9.445,71.629
2,10.0,9.584,66.204
3,100.0,13.157,41.046
